#### Data extraction
1. Loading SrpWN excel file
2. Extracting serban and english sheets in different datasets
3. Comparing the sysnet IDs, serching for untraslated ones and creating database of untraslated synsets - candidates for adding to SrpwN

In [ ]:
#Import libs
import pandas as pd
import numpy as np

In [ ]:
#Load srpwn excel file
srpwn_med_filename = "./data/srpwn - medicina.xlsx"

xls = pd.ExcelFile(srpwn_med_filename)

srpwn = pd.read_excel(xls, sheet_name="srpski")
wordnet = pd.read_excel(xls, sheet_name="engleski")

In [ ]:
srpwn.describe()

In [ ]:
wordnet.describe()

In [ ]:
#Number of synsets in SrpWN 
print("Number of synsets in SrpWN:", len(srpwn))
print(srpwn.shape)

#Number of synsets in WordNet
print("Number of synsets in WordNet:", len(wordnet))
print(wordnet.shape)

In [ ]:
len(wordnet.ID.unique())

In [ ]:
len(srpwn.ID.unique())

In [ ]:
sum(srpwn.ID.value_counts()>1) 

In [ ]:
sum(wordnet.ID.value_counts()>1)

Searching for candidates

In [ ]:
#Only in SrpWN
srpwn_only = pd.merge(srpwn, wordnet, on='ID', how='left_anti')
print("Only in SrpWN =", len(srpwn_only))

#Only in WordNet - number of candidates for adding to srpwn
# srpwn_candidates = pd.merge(wordnet, srpwn, on='ID', how='left_anti')
srpwn_candidates = wordnet[~wordnet["ID"].isin(srpwn["ID"])]

print("Number of untranslated records:", len(srpwn_candidates))

In [ ]:
#Detection of missing vales
#It is noted that some columns have symbol "-" as empty value those values as Nan

srpwn.replace("-", np.nan, inplace=True)
print(srpwn.isna().sum())

srpwn.columns[pd.isna(srpwn).any()].tolist()

In [ ]:
def empty(value):
    """Returns True if the cell is empty, NaN or whitespace."""
    if pd.isna(value):
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False
 

In [ ]:
srpwn_ids = set(srpwn.ID)
wordnet_ids = set(wordnet.ID)

matched_ids = wordnet_ids & srpwn_ids
unmatched_ids  = wordnet_ids - srpwn_ids

print("Synsets missing in srpwn:", len(unmatched_ids))
print("Sysnets matched in both sheets:", len(matched_ids))
 

In [ ]:
# print(wordnet[wordnet.ID.duplicated()==True].ID)
duplicates = wordnet.ID.value_counts()
print("Duplicate synsets in WordNet:", (duplicates>1).sum())

print("Unique synsets in WordNet:" , len(wordnet.ID.unique()))
print("In WordNet:" , len(wordnet))

#### Cuvanje kandidata u poseban excel fajl na kom se nastavlja pilot test modela i razlicitih prompt strategija

In [ ]:
# --- Snimi detaljne rezultate za dalju upotrebu ---
pilot = srpwn_candidates.sample(n=200, random_state=42)
pilot.to_csv("./data/nedostajuci_sinsetovi.csv", index=False)


print("\nSnimljeno:")
print("  - nedostajuci_sinsetovi.csv")

 

In [ ]:
len(wordnet) - (duplicates>1).sum()

In [ ]:
pilot.shape

In [ ]:
# Matched IDs, synset which has both english and serbian translation, it will be used for verify tranlsation quality
already_translated_synsets = wordnet[wordnet["ID"].isin(srpwn["ID"])]
already_translated_synsets.to_csv("./data/SrpWN_translated_synsets.csv", index = False)

print("\nSnimljeno:")
print("  - SrpWN_translated_synsets.csv")